# PointNet++ MATLAB PCD Finetuning
This notebook resumes training from your 10th epoch SemanticKITTI checkpoint, but exclusively trains on your friend's MATLAB `.pcd` dataset.

**Pre-requisite:** You must add a shortcut of the shared `LIDAR SIH` folder to your main Google Drive so Colab can see it! Right-click the folder in `Shared with me` and click `Organize -> Add shortcut` -> `My Drive`.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Setup PointNet++ Repository and Install Dependencies
import os
if not os.path.exists('/content/Pointnet_Pointnet2_pytorch'):
    !git clone https://github.com/yanx27/Pointnet_Pointnet2_pytorch.git
    
!pip install -q pypcd4 pandas matplotlib tqdm

In [ ]:
%%writefile matlab_pcd_dataset.py
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset
from pypcd4 import PointCloud

class MatlabPCDDataset(Dataset):
    def __init__(self, data_path, num_points=4096, split='train'):
        self.data_path = data_path
        self.num_points = num_points
        self.split = split
        
        # Map MATLAB class IDs directly to the 20 classes PointNet++ learned from SemanticKITTI.
        # Anything not in this dictionary gets mapped to 0 (unlabeled) and is ignored during training.
        self.matlab_to_kitti_map = {
            1: 1,   # Car -> car
            2: 4,   # Truck -> truck
            3: 2,   # Bicycle -> bicycle
            4: 6,   # Pedestrian -> person
            5: 14,  # Jersey Barrier -> fence
            6: 14   # Guardrail -> fence
        }
        
        # Recursively find all .pcd files in all subfolders of the dataset path
        self.pcd_files = sorted(glob.glob(os.path.join(data_path, '**', '*.pcd'), recursive=True))
        
        if len(self.pcd_files) == 0:
            print(f"Warning: No .pcd files found in {data_path}!")
            
        # Simple train/val split (80/20)
        split_idx = int(len(self.pcd_files) * 0.8)
        if self.split == 'train':
            self.pcd_files = self.pcd_files[:split_idx]
        else:
            self.pcd_files = self.pcd_files[split_idx:]
            
        print(f"Loaded {len(self.pcd_files)} PCD frames for {self.split} split.")

    def __len__(self):
        return len(self.pcd_files)

    def __getitem__(self, idx):
        pcd_path = self.pcd_files[idx]
        pc = PointCloud.from_path(pcd_path)
        
        # 1. Extract XYZ and Intensity
        try:
            scan = pc.numpy(['x', 'y', 'z', 'intensity']).astype(np.float32)
        except Exception as e:
            xyz = pc.numpy(['x', 'y', 'z']).astype(np.float32)
            scan = np.column_stack((xyz, np.zeros((xyz.shape[0], 1), dtype=np.float32)))
            
        # 2. Extract Labels (class_id)
        try:
            # pypcd4 extracts it as a 1D array if given a string, or 2D if list. 
            # We just want a 1D array of integers.
            raw_labels = pc.pc_data['class_id'].astype(np.int32)
        except Exception as e:
            print(f"Warning: 'class_id' field missing in {pcd_path}. Defaulting to unlabeled.")
            raw_labels = np.zeros(scan.shape[0], dtype=np.int32)
            
        # 3. Filter out NaN points to prevent CUDA crashes
        valid_mask = ~np.isnan(scan).any(axis=1) & np.isfinite(scan).all(axis=1)
        scan = scan[valid_mask]
        raw_labels = raw_labels[valid_mask]
        
        # 4. Normalize Intensity (if it's 0-255)
        if np.max(scan[:, 3]) > 1.0:
            scan[:, 3] = scan[:, 3] / 255.0
            
        # 5. Zero-Center Coordinates to prevent Float32 precision errors in PyTorch distance calculations
        centroid = np.mean(scan[:, :3], axis=0)
        scan[:, :3] = scan[:, :3] - centroid
        
        # 6. Map MATLAB labels to SemanticKITTI labels
        mapped_labels = np.zeros_like(raw_labels)
        for mat_id, kitti_id in self.matlab_to_kitti_map.items():
            mapped_labels[raw_labels == mat_id] = kitti_id
            
        # 7. Sample 4096 points
        num_raw = scan.shape[0]
        if num_raw >= self.num_points:
            choice = np.random.choice(num_raw, self.num_points, replace=False)
        else:
            choice = np.random.choice(num_raw, self.num_points, replace=True)
            
        sampled_scan = scan[choice, :]
        sampled_label = mapped_labels[choice]
        
        # 8. Convert to PyTorch Tensors
        point_features = torch.tensor(sampled_scan, dtype=torch.float32).transpose(0, 1) # [4, 4096]
        point_labels = torch.tensor(sampled_label, dtype=torch.long)                     # [4096]
        
        return point_features, point_labels


In [ ]:
%%writefile outdoor_pointnet.py
import os
import sys
import torch
import torch.nn as nn

repo_path = os.path.abspath('Pointnet_Pointnet2_pytorch')
if repo_path not in sys.path:
    sys.path.append(repo_path)

import models.pointnet2_sem_seg as pointnet2_sem_seg
from models.pointnet2_utils import PointNetSetAbstraction

def get_outdoor_model(num_classes=20, input_channels=1):
    model = pointnet2_sem_seg.get_model(num_classes)
    
    # Note: input_channels+6 because the repo passes all 3 coordinates PLUS the 4 feature channels into sa1.
    model.sa1 = PointNetSetAbstraction(1024, 0.1, 32, input_channels + 6, [32, 32, 64], False)
    model.conv2 = nn.Conv1d(128, num_classes, 1)
    return model

def load_pretrained_weights(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, weights_only=False)
    # If the checkpoint contains 'model_state_dict' (like the pre-trained weights from the repo), extract it.
    # Otherwise, if it was saved by our script directly, the checkpoint IS the state dict!
    if 'model_state_dict' in checkpoint:
        pretrained_dict = checkpoint['model_state_dict']
    else:
        pretrained_dict = checkpoint
        
    model_dict = model.state_dict()
    
    filtered_dict = {k: v for k, v in pretrained_dict.items() if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(filtered_dict)
    model.load_state_dict(model_dict)
    return model


In [ ]:
%%writefile finetune_matlab.py
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

from matlab_pcd_dataset import MatlabPCDDataset
from outdoor_pointnet import get_outdoor_model, load_pretrained_weights

def main():
    # --- DATASET PATH ---
    # Assuming you made a shortcut of 'LIDAR SIH' to your main Google Drive.
    DATASET_PATH = '/content/drive/MyDrive/LIDAR SIH/Ryan'
    
    # --- CHECKPOINT TO RESUME FROM ---
    # We are resuming from your fine-tuned SemanticKITTI checkpoint (epoch 10)!
    RESUME_CHECKPOINT = '/content/drive/MyDrive/checkpoints_kitti/semantickitti_epoch_10.pth'
    
    BATCH_SIZE = 4
    NUM_POINTS = 4096
    NUM_EPOCHS = 10
    LEARNING_RATE = 1e-4
    NUM_CLASSES = 20
    INPUT_CHANNELS = 1
    
    if not os.path.exists(DATASET_PATH):
        print(f"ERROR: Dataset not found at {DATASET_PATH}. Please make sure you added a shortcut to your Drive!")
        return

    print("Initializing MATLAB PCD datasets...")
    train_dataset = MatlabPCDDataset(DATASET_PATH, num_points=NUM_POINTS, split='train')
    val_dataset = MatlabPCDDataset(DATASET_PATH, num_points=NUM_POINTS, split='val')
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
    
    print("Initializing Outdoor PointNet++...")
    model = get_outdoor_model(num_classes=NUM_CLASSES, input_channels=INPUT_CHANNELS)
    
    if os.path.exists(RESUME_CHECKPOINT):
        print(f"Loading your previous weights from {RESUME_CHECKPOINT}...")
        model = load_pretrained_weights(model, RESUME_CHECKPOINT)
    else:
        print(f"WARNING: Could not find {RESUME_CHECKPOINT}. Starting from scratch!")
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    class_weights = torch.ones(NUM_CLASSES).to(device) 
    class_weights[0] = 0.0 # Ignore unlabeled points
    
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    
    print("Starting Finetuning Loop on MATLAB Dataset...")
    for epoch in range(NUM_EPOCHS):
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
        for batch_features, batch_labels in pbar:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)
            
            optimizer.zero_grad()
            predictions, _ = model(batch_features)
            predictions = predictions.transpose(1, 2)
            loss = criterion(predictions, batch_labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
            
        avg_train_loss = train_loss / len(train_loader)
        print(f"Epoch {epoch+1} - Average Train Loss: {avg_train_loss:.4f}")
        
        # Save these NEW checkpoints in a separate folder so they don't overwrite your KITTI weights!
        os.makedirs('/content/drive/MyDrive/checkpoints_matlab', exist_ok=True)
        torch.save(model.state_dict(), f'/content/drive/MyDrive/checkpoints_matlab/matlab_epoch_{epoch+1}.pth')

if __name__ == '__main__':
    main()


In [ ]:
# Execute the fine-tuning script!
!python finetune_matlab.py